# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "11-01-2021"
end_date = "07-04-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"
metadata_folder = originals + "avian-influenza/metadata/"

## Read Metadata 

In [3]:
# Read metadata

# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated.csv")
print(len(metadata)) 

# If metadata_normalized.tsv is updated, merge to get collection dates
# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates
# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

print(len(metadata)) 
# display(metadata)

9899
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
9899


In [4]:
# Get list of genotypes

# os.chdir(references)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1", "D1.3"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name = genbank_mapping.tsv > genbank_name

geo_location = geo_loc_name (abbreviated)-country (abbreviated) e.g. USA-MD

isolate = isolate

collection date (primary) = Collection_Date

collection date = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = H5N1 (hard-coded)

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

## Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
# display(metadata)

7706


## Get specific geolocation

In [6]:
# Get specific geolocation and name_state from genbank_mapping.tsv
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping = genbank_mapping.rename(columns={"sra_run": "Run"}) # Rename so we can merge
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates -- there are ~8 copies of each run
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# Merge with metadata so that we can have specific geolocation
metadata_genbank = pd.concat([metadata, genbank_mapping], join="inner") # Exclude runs without geolocation

# print(metadata_genbank)

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
# Format: USA-[state abbreviation], e.g. USA-MD
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(', ', ',').replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(', ', ',').replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ',').replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ',').replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(', ', ',').replace(" ", "_").split(','))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        else 
                                                        x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata[metadata["Geo_Location"] != "USA"]) 

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state_y,Geo_Location
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,Texas,USA-TX
3,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,2025-05-09_10-45-55,SRR28752447.fa,B3.13,"NP:am8, MP:ea1, PB2:am2.2, HA:ea1, NA:ea1, PA:...","am8:23-032005-001:NP, ea1:22-003707-003:MP, am...","99.40%, 98.88%, 98.86%, 98.77%, 99.08%, 99.16%...","9, 11, 26, 21, 13, 18, 6, 10",Ran on FASTA - No Coverage Report,Texas,USA-TX
5,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,2025-05-09_10-47-06,SRR28752448.fa,B3.13,"HA:ea1, NA:ea1, MP:ea1, PA:ea1, PB1:am4, PB2:a...","ea1:22-003707-003:HA, ea1:22-003707-003:NA, ea...","98.77%, 99.08%, 98.88%, 99.21%, 99.56%, 98.86%...","21, 13, 11, 17, 10, 26, 7, 10",Ran on FASTA - No Coverage Report,Texas,USA-TX
7,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,2025-05-09_10-46-33,SRR28752449.fa,B3.13,"MP:ea1, PA:ea1, PB2:am2.2, NP:am8, NS:am1.1, N...","ea1:22-003707-003:MP, ea1:22-003707-003:PA, am...","98.88%, 99.16%, 98.86%, 99.33%, 99.28%, 99.08%...","11, 18, 26, 10, 6, 13, 21, 10",Ran on FASTA - No Coverage Report,Texas,USA-TX
9,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,2025-05-09_10-47-05,SRR28752450.fa,B3.13,"PA:ea1, PB2:am2.2, MP:ea1, NS:am1.1, PB1:am4, ...","ea1:22-003707-003:PA, am2.2:22-010445-001:PB2,...","99.21%, 98.82%, 98.88%, 99.17%, 99.56%, 98.77%...","17, 27, 11, 7, 10, 21, 10, 13",Ran on FASTA - No Coverage Report,Texas,USA-TX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12225,SRR33764503,WGS,148.07,72144967,PRJNA980729,SAMN48808512,Viral,23163428,USDA-NVSL,2025,...,2025-05-31_08-53-49,SRR33764503.fa,D1.1,"PB1:ea3, NP:am13, MP:ea3, HA:ea3, PB2:am24, NA...","ea3:22-013001-001:PB1, am13:24-030039-001:NP, ...","99.08%, 99.27%, 99.69%, 99.35%, 99.52%, 98.95%...","21, 11, 3, 11, 11, 11, 13, 8",Ran on FASTA - No Coverage Report,AZ,USA-AZ
12227,SRR33764504,WGS,147.97,84292892,PRJNA980729,SAMN48808511,Viral,26419266,USDA-NVSL,2025,...,2025-05-31_08-53-49,SRR33764504.fa,D1.1,"NA:am4N1, NP:am13, PA:am4, MP:ea3, PB2:am24, P...","am4N1:24-030039-001:NA, am13:24-030039-001:NP,...","98.95%, 99.40%, 99.49%, 100.00%, 99.56%, 99.21...","11, 9, 11, 0, 10, 18, 10, 9",Ran on FASTA - No Coverage Report,AZ,USA-AZ
12229,SRR33764505,WGS,147.58,73626165,PRJNA980729,SAMN48808510,Viral,23101168,USDA-NVSL,2025,...,2025-05-31_08-53-49,SRR33764505.fa,D1.1,"NA:am4N1, MP:ea3, NS:ea3, PA:am4, HA:ea3, PB2:...","am4N1:24-030039-001:NA, ea3:22-013001-001:MP, ...","98.85%, 100.00%, 98.93%, 99.49%, 99.41%, 99.56...","12, 0, 9, 11, 10, 10, 18, 9",Ran on FASTA - No Coverage Report,AZ,USA-AZ
12231,SRR33764506,WGS,145.88,25188698,PRJNA980729,SAMN48808501,Viral,7754717,USDA-NVSL,2025,...,2025-05-31_08-53-48,SRR33764506.fa,D1.1,"PB1:ea3, PA:am4, PB2:am24, NP:am13, NS:ea3, NA...","ea3:22-013001-001:PB1, am4:24-030039-001:PA, a...","99.25%, 99.49%, 99.52%, 99.60%, 98.93%, 99.04%...","17, 9, 11, 6, 9, 10, 9, 0",Ran on FASTA - No Coverage Report,AZ,USA-AZ


## Collection Dates

If date is N/A, try finding it first. If a csv file of saved dates (NOT metadata_normalized.tsv) are available, do NOT run the next cell. Comment it out and run the cell after. 

In [7]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x) # Real dates have dashes
# # Convert dates to date format
# try:
#     metadata["Collection_Date_Specific"] = metadata["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x, default=datetime(1, 1, 2000), fuzzy=True))
# except:
#     print("Unable to parse collection date.")

If saved dates are available, un-comment and run the next cell

In [8]:
# Upload saved data -- if doing this, make sure the above cell is commented out
os.chdir(temp_files)
metadata_genbank = pd.read_csv("metadata_genbank_11-01-2021--06-13-2025.csv")

# Get only updated dates

def find_unknown_dates(x, df):
    try:
        date = metadata_genbank[metadata_genbank["BioSample"] == x]["Collection_Date"].values[0]
        date = dateutil.parser.parse(date, default=datetime(2000, 1, 1), fuzzy=True) # Default is January 1st, 2000
        if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
            print("year only")
            date = search_collection_date(x, df)
        print("Success", date)
    except:
        date = search_collection_date(x, df) # If it's not parseable as a date

    # Ensure that the date is converted to date format
    try:
        # date = dateutil.parser.parse(date, default=datetime(2000, 1, 1), fuzzy=True)
        # print("yay")
        if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
            date = date.year.strftime("%Y") # Preserve only year
        else: # If actual date
            date = date.strftime("%Y-%m-%d")
        print("Successfully parsed", date)
    except:
        print("Unable to parse date.")

    return date # "If" statement in lambda function will search for the "just year" values

updated_dates = metadata["BioSample"].apply(lambda x: find_unknown_dates(x, metadata)) # Update unknown dates, if possible
metadata["Collection_Date_Specific"] = updated_dates

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_99728\1743879338.py:3: DtypeWarning: Columns (40,45,47) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_genbank = pd.read_csv("metadata_genbank_11-01-2021--06-13-2025.csv")


Success 2024-03-16 00:00:00
Successfully parsed 2024-03-16
Success 2024-03-16 00:00:00
Successfully parsed 2024-03-16
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-20
Success 2024-03-13 00:00:00
Successfully parsed 2024-03-13
Success 2024-03-13 00:00:00
Successfully parsed 2024-03-13
Success 2024-03-13 00:00:00
Successfully parsed 2024-03-13
Success 2024-03-13 00:00:00
Successfully parsed 2024-03-13
Success 2024-03-20 00:00:00
Successfully parsed 2024-03-

In [9]:
# Save the above so we don't have to do it again
os.chdir(temp_files)
metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# Get years from collection dates
metadata["years"] = metadata["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x, default=datetime(2000, 1, 1), fuzzy=True).year) # Get year only from collection date

print(metadata["Collection_Date_Specific"])

TypeError: Parser must be a string or character stream, not float

## Get host type

In [ ]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x # If NaN
                                                      or "/" not in x # If split isolate doesn't exist 
                                                      or len(x.split("/")) < 2 # If split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
                    avian               cattle        feline   other_mammal  \
0        great_horned_owl            dairy_cow           cat     deer mouse   
1            common_raven               cattle  domestic_cat    house_mouse   
2           cooper's_hawk  cattle milk product     feral_cat          skunk   
3            coopers_hawk          bovine_milk        feline  striped_skunk   
4                 peafowl              bovine   domestic-cat     norway rat   
..                    ...                  ...           ...            ...   
799         harris's_hawk                  NaN           NaN            NaN   
800         eurasian_coot                  NaN           NaN            NaN   
801                 layer                  NaN           NaN            NaN   
802            perdicinae                  NaN           NaN            NaN   
803  great-tailed grackle                  NaN           NaN            NaN   

          human         other  new  
0    washin

In [ ]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

## Make names using all the attributes we collected

In [ ]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["name_state_y"] + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date_Specific"].apply(lambda x: x if "-" not in x else str(x.strftime("%Y")) if x.month == datetime(2000, 1, 1).month and x.day == datetime(2000, 1, 1).day else x.strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

0      >SRR33993023|A/cattle/USA/25-016878-002/2025|H...
1      >SRR33993024|A/cattle/USA/25-016866-004/2025|H...
2      >SRR33993025|A/cattle/USA/25-016866-002/2025|H...
3      >SRR33993026|A/cattle/USA/25-016847-007/2025|H...
4      >SRR33993027|A/cattle/USA/25-016847-006/2025|H...
                             ...                        
111    >SRR34270140|A/duck/USA/25-015646-002/2025|H5N...
112    >SRR34270141|A/chicken/USA/25-015646-001/2025|...
113    >SRR34270142|A/chicken/USA/25-010529-003/2025|...
114    >SRR34270143|A/chicken/USA/25-010529-002/2025|...
115    >SRR34270144|A/duck/USA/25-010529-001/2025|H5N...
Name: Name, Length: 116, dtype: object

In [ ]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [ ]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR33993023        WGS      148.15   68132847  PRJNA1102327   
1    SRR33993024        WGS      148.15   63105544  PRJNA1102327   
2    SRR33993025        WGS      148.28   46180111  PRJNA1102327   
3    SRR33993026        WGS      147.53   47042737  PRJNA1102327   
4    SRR33993027        WGS      147.76   65389924  PRJNA1102327   
..           ...        ...         ...        ...           ...   
111  SRR34270140        WGS      148.74  131194009   PRJNA980729   
112  SRR34270141        WGS      148.49  110079404   PRJNA980729   
113  SRR34270142        WGS      146.95  112507279   PRJNA980729   
114  SRR34270143        WGS      148.02  142063798   PRJNA980729   
115  SRR34270144        WGS      147.58  379458191   PRJNA980729   

        BioSample BioSampleModel      Bytes Center Name Collection_Date  ...  \
0    SAMN49104730          Viral   22640539   USDA-NVSL            2025  ...   
1    SAMN49104729      

## Make FASTA files

In [ ]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [ ]:
# Create fasta files 

os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty
        output_path = originals + "complete/" + pair + "_" + date_range + "_andersen.fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR33993023|A/cattle/USA/25-016878-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993024|A/cattle/USA/25-016866-004/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993025|A/cattle/USA/25-016866-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993026|A/cattle/USA/25-016847-007/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993027|A/cattle/USA/25-016847-006/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993028|A/cattle/USA/25-016847-004/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993041|A/cattle/USA/25-016745-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993043|A/cattle/USA/25-017370-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993045|A/cattle/USA/25-017364-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993046|A/cattle/USA/25-017364-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993048|A/cattle/USA/25-002509-001-tile/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993049|A/cattle/USA/25-017049-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993050|A/cattle/USA/25-016878-005/2025|H5N1|USA|2025|cattle|B3.13
>SRR33993051|A/cattle/USA/25-016878-003/2025|H5N1|USA|2025|